# Brasil — soma dos oito inversores por instante

Resolução real de **cinco minutos**, horizonte **t+5 minutos**. Os nomes dos arquivos foram mantidos por solicitação do usuário; o conteúdo foi migrado de diário para intradiário.

O alvo e a persistência (t−5 min) precisam ser originalmente medidos. Cada etapa de preenchimento causal usa até 15 minutos; no Brasil a combinação proxy + ffill pode atingir 30 minutos somente no histórico: nunca se interpola usando ponto posterior. Não se concatenam blocos interrompidos. A avaliação principal é diurna (elevação solar > 0); a avaliação 24h é complementar. Zero noturno não deve inflar a precisão.

As entradas principais são potência AC em p.u., hora local e dia do ano em seno/cosseno. O vento continua excluído. ERA5 nativa horária é auxiliar e não entra nos modelos principais. Estatísticas de normalização vêm apenas do treino australiano; alvos de validação/teste não ajustam escalas.


In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
from IPython.display import display,Image,Markdown
RAIZ=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/energia_transfer').is_dir())
if str(RAIZ/'src') not in sys.path: sys.path.insert(0,str(RAIZ/'src'))
PASTA=RAIZ/'experimentos/transfer_learning_australia_brasil'
AU=RAIZ/'experimentos/australia_deep_learning'
REPORT=RAIZ/'reports/intradiario'
from energia_transfer.transfer_learning.dados import carregar,particao
_,CONFIG,CONTRATO=carregar(PASTA)
def exibir(t): display(t.round(5))


In [2]:
df=pd.read_pickle(RAIZ/'data/processed/brasil/base_5min_brasil.pkl')
exibir(df.head(15))
print('Alvos medidos:',df.Active_Power_original.notna().sum())
print('Históricos reconstruídos:',df.potencia_estimada.sum())
print('Grade:',df.index.to_series().diff().dropna().value_counts().to_dict())


Alvos medidos: 90023
Históricos reconstruídos: 12627
Grade: {Timedelta('0 days 00:05:00'): 105119}


,Active_Power_original,Active_Power,registro_original,diurno,inversores_medidos,inversores_proxy,inversores_zero_fisico,potencia_estimada,Weather_Temperature_Celsius,Weather_Relative_Humidity,Global_Horizontal_Radiation,hora_sin,hora_cos,dia_ano_sin,dia_ano_cos,meteo_resolucao_nativa_min,meteo_idade_min,temperatura_estacao_original,ghi_estacao_original_escala_nao_validada
timestamp,,,,,,,,,,,,,,,,,,,
2025-01-01 00:00:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.00000,1.00000,0.0,1.0,60,0,NaN,NaN
2025-01-01 00:05:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.02181,0.99976,0.0,1.0,60,5,NaN,NaN
2025-01-01 00:10:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.04362,0.99905,0.0,1.0,60,10,NaN,NaN
2025-01-01 00:15:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.06540,0.99786,0.0,1.0,60,15,NaN,NaN
2025-01-01 00:20:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.08716,0.99619,0.0,1.0,60,20,NaN,NaN
2025-01-01 00:25:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.10887,0.99406,0.0,1.0,60,25,NaN,NaN
2025-01-01 00:30:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.13053,0.99144,0.0,1.0,60,30,NaN,NaN
2025-01-01 00:35:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.15212,0.98836,0.0,1.0,60,35,NaN,NaN
2025-01-01 00:40:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.17365,0.98481,0.0,1.0,60,40,NaN,NaN


In [3]:
resumo=json.loads((REPORT/'resumo.json').read_text(encoding='utf-8'))
print(json.dumps(resumo,ensure_ascii=False,indent=2))


{
  "resolucao": "5min",
  "horizonte_minutos": 5,
  "linhas_australia_brutas": 1256431,
  "duplicatas_australia": 0,
  "australia": {
    "grade": 1270529,
    "alvos_medidos": 1210930,
    "inputs_estimados": 39596
  },
  "brasil": {
    "grade": 105120,
    "alvos_medidos": 90023,
    "inputs_estimados": 12627,
    "estacao_temperatura_presente": 33383
  },
  "alvos_imputados": 0,
  "meteo_brasil": "ERA5 horária carregada para frente até 55 minutos; somente auxiliar, excluída do contrato principal",
  "reconstrucao": "Proxy causal e ffill de até três passos nos inputs; sem interpolação com ponto posterior. Alvos exigem medição original.",
  "fontes_sha256": {
    "australia": "adebe983a30f58c5900247308790ed0659de9fdff2a4fd59f6e464147e3c2d14",
    "brasil": "598ede357558a5655894945c9d3aed845dd3fb5ed9e61e25d1f70cbec8611314",
    "estacao": "c51aaf7f7486c8af703bbaaa01c99105b1c5cbd3d73a9543ee93494fae3158be"
  }
}
